# CredX - Step 2: Feature Engineering
Calculates 9 alternative financial behavior signals:
1. `avg_income`: 6-month mean
2. `income_std`: 6-month standard deviation
3. `income_growth`: % delta from Month 1 to Month 6
4. `income_stability_score`: 100 * (1 - (income_std / avg_income))
5. `utility_payment_rate`: utility_bills_paid / utility_bills_total
6. `rent_reliability_rate`: rent_paid_on_time_months / total_rental_months
7. `payment_reliability_score`: 50% utility rate + 50% rent rate (0-100)
8. `mobile_years`: 2026 - same_number_since_year
9. `digital_trust_score`: MinMax weighted composite (0-100)


In [ ]:
import pandas as pd
import numpy as np

CLEAN_PATH = "../data/processed/cleaned_dataset.csv"
df = pd.read_csv(CLEAN_PATH)
print("Cleaned Dataset Shape:", df.shape)


## 1. Income Dynamics Features


In [ ]:
income_cols = [f'income_month_{i}' for i in range(1, 7)]

# 1. Average Income
df['avg_income'] = df[income_cols].mean(axis=1)

# 2. Income Standard Deviation
df['income_std'] = df[income_cols].std(axis=1)

# 3. Income Growth
denom = df['income_month_1'].replace(0, np.nan)
df['income_growth'] = (((df['income_month_6'] - df['income_month_1']) / denom) * 100).clip(-100, 300)

# 4. Income Stability Score (0-100)
cov = (df['income_std'] / df['avg_income']).fillna(1.0)
df['income_stability_score'] = (100.0 * (1.0 - cov)).clip(0, 100)

df[['avg_income', 'income_std', 'income_growth', 'income_stability_score']].describe().T


## 2. Payment Discipline Features


In [ ]:
# 5. Utility Payment Rate
df['utility_payment_rate'] = (df['utility_bills_paid'] / df['utility_bills_total'].replace(0, np.nan)).fillna(1.0).clip(0, 1)

# 6. Rent Reliability Rate (non-renters inherit utility rate)
is_renter = df['total_rental_months'] > 0
df['rent_reliability_rate'] = np.where(
    is_renter,
    (df['rent_paid_on_time_months'] / df['total_rental_months'].replace(0, np.nan)).fillna(1.0).clip(0, 1),
    df['utility_payment_rate']
)

# 7. Payment Reliability Score (0-100)
df['payment_reliability_score'] = ((0.50 * df['utility_payment_rate'] + 0.50 * df['rent_reliability_rate']) * 100).clip(0, 100)

df[['utility_payment_rate', 'rent_reliability_rate', 'payment_reliability_score']].describe().T


## 3. Mobile Stability & Digital Trust Score


In [ ]:
CURRENT_YEAR = 2026

# 8. Mobile Years
df['mobile_years'] = (CURRENT_YEAR - df['same_number_since_year']).clip(0, 30)

# 9. Digital Trust Score (MinMax normalization across 5 behavioral signals)
def minmax(series):
    q99 = series.quantile(0.99)
    min_v = series.min()
    return ((series - min_v) / (q99 - min_v)).clip(0, 1) * 100.0

s_upi_tx = minmax(df['upi_transactions_per_month'])
s_upi_m = minmax(df['upi_months_active'])
s_my = minmax(df['mobile_years'])
s_w = minmax(df['mobile_wallet_used'])
s_rf = minmax(df['recharge_frequency_per_month'])

df['digital_trust_score'] = (
    0.30 * s_upi_tx +
    0.25 * s_upi_m +
    0.20 * s_my +
    0.15 * s_w +
    0.10 * s_rf
).clip(0, 100)

df[['mobile_years', 'digital_trust_score']].describe().T


## 4. Export Featured Dataset


In [ ]:
OUTPUT_PATH = "../data/processed/featured_dataset.csv"
df.to_csv(OUTPUT_PATH, index=False)
print("Exported featured dataset to:", OUTPUT_PATH)
print("Final Shape:", df.shape)
